<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Run Cosmos3 Nano and Super FP8 checkpoints with TensorRT-LLM

TensorRT-LLM can run ModelOpt-calibrated Cosmos3 Nano and Super FP8
checkpoints directly. The checkpoint metadata supplies the calibrated weight
and activation scales, so no quantization CLI flag is
needed.

This notebook covers the four validated single-GPU modes:

| Mode                 | Prompt or condition                   | Output    |
| -------------------- | ------------------------------------- | --------- |
| Text to image (T2I)  | Structured text prompt                | PNG image |
| Text to video (T2V)  | Structured text prompt                | MP4 video |
| Image to video (I2V) | Structured prompt and reference image | MP4 video |
| Video to video (V2V) | Structured prompt and reference video | MP4 video |

FP8 Cosmos3 is single-GPU only. TensorRT-LLM rejects tensor parallelism,
Ulysses parallelism, context parallelism, CFG parallelism, or parallel VAE
sizes greater than one for these checkpoints. Use a BF16 checkpoint for a
multi-GPU deployment.

## 1. Prerequisites

Build and install TensorRT-LLM from its `main` branch by following the
[TensorRT-LLM source-build guide](https://nvidia.github.io/TensorRT-LLM/installation/build-from-source.html).
The source checkout supplies the Cosmos3 example, prompt files, and one-GPU
configuration files used below.

Install `ffmpeg` so TensorRT-LLM can encode MP4 output. Install the Cosmos3
guardrail package, accept the gated
[`nvidia/Cosmos-1.0-Guardrail`](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail)
terms, and authenticate to Hugging Face before generation:

```shell
apt-get update && apt-get install -y ffmpeg
pip install cosmos_guardrail==0.3.0
pip uninstall -y opencv-python
pip install opencv-python-headless
hf auth login
```

The pinned guardrail package performs prompt text checks and RetinaFace face
blurring. Its generated-frame content classifier is disabled upstream because
it produced too many false positives.

Nano and Super publish their ModelOpt FP8 checkpoints on the `fp8` revision of
their Hugging Face repositories. Download each revision to a local directory;
TensorRT-LLM receives that directory through `--model`:

```shell
hf download nvidia/Cosmos3-Nano \
  --revision fp8 \
  --local-dir checkpoints/Cosmos3-Nano-FP8
hf download nvidia/Cosmos3-Super \
  --revision fp8 \
  --local-dir checkpoints/Cosmos3-Super-FP8
```

Set the paths used by the commands in this notebook. You may export these
variables before starting Jupyter or replace the placeholder defaults below:



In [ ]:
from pathlib import Path
import os


TRTLLM_ROOT = Path(os.environ.get("TRTLLM_ROOT", "/path/to/TensorRT-LLM")).expanduser().resolve()
COSMOS3_NANO_FP8 = Path(
    os.environ.get("COSMOS3_NANO_FP8", Path.cwd() / "checkpoints" / "Cosmos3-Nano-FP8")
).expanduser().resolve()
COSMOS3_SUPER_FP8 = Path(
    os.environ.get("COSMOS3_SUPER_FP8", Path.cwd() / "checkpoints" / "Cosmos3-Super-FP8")
).expanduser().resolve()
COSMOS3_FP8_OUTPUTS = Path(
    os.environ.get("COSMOS3_FP8_OUTPUTS", Path.cwd() / "outputs" / "tensorrt_llm_fp8")
).expanduser().resolve()

for name, path in {
    "TRTLLM_ROOT": TRTLLM_ROOT,
    "COSMOS3_NANO_FP8": COSMOS3_NANO_FP8,
    "COSMOS3_SUPER_FP8": COSMOS3_SUPER_FP8,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"Set {name} to an existing path; got {path}")

COSMOS3_FP8_OUTPUTS.mkdir(parents=True, exist_ok=True)
for name, path in {
    "TRTLLM_ROOT": TRTLLM_ROOT,
    "COSMOS3_NANO_FP8": COSMOS3_NANO_FP8,
    "COSMOS3_SUPER_FP8": COSMOS3_SUPER_FP8,
    "COSMOS3_FP8_OUTPUTS": COSMOS3_FP8_OUTPUTS,
}.items():
    os.environ[name] = str(path)
    print(f"{name}={path}")



The checkpoints may carry a `diffusion_step_policy` in their quantization
metadata. TensorRT-LLM applies that checkpoint-owned policy automatically; do
not add a separate runtime override for it.

## 2. Cosmos3 Nano FP8

### Text to image

The text-to-image config warms the 1024x1024, one-frame shape. The structured
prompt selects image mode, and `--output_type image` makes the expected output
explicit.



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_NANO_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-t2i-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/t2i.json" \
  --output_type image \
  --output_path "$COSMOS3_FP8_OUTPUTS/nano_t2i.png"



### Text to video



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_NANO_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/t2v.json" \
  --output_path "$COSMOS3_FP8_OUTPUTS/nano_t2v.mp4"



### Image to video

The example I2V prompt names its reference image with an HTTPS URL. Use
`--image_path` to replace it with a local path, `file://` URL, HTTP(S) URL, or
`data:` URI.



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_NANO_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/i2v.json" \
  --output_path "$COSMOS3_FP8_OUTPUTS/nano_i2v.mp4"



### Video to video

This command uses the Nano T2V output above as the reference video. By default,
the first five decoded pixel frames condition the output.



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_NANO_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/v2v.json" \
  --video_path "$COSMOS3_FP8_OUTPUTS/nano_t2v.mp4" \
  --output_path "$COSMOS3_FP8_OUTPUTS/nano_v2v.mp4"



## 3. Cosmos3 Super FP8

The same one-GPU interfaces apply to the Super checkpoint. Do not use
`cosmos3-super-4gpu.yaml`; these FP8 checkpoints reject that parallel
configuration.

### Text to image



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_SUPER_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-t2i-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/t2i.json" \
  --output_type image \
  --output_path "$COSMOS3_FP8_OUTPUTS/super_t2i.png"



### Text to video

The file name `cosmos3-nano-1gpu.yaml` is historical: TensorRT-LLM documents
the configuration as the shared one-GPU config for both Nano and Super.



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_SUPER_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/t2v.json" \
  --output_path "$COSMOS3_FP8_OUTPUTS/super_t2v.mp4"



### Image to video



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_SUPER_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/i2v.json" \
  --output_path "$COSMOS3_FP8_OUTPUTS/super_i2v.mp4"



### Video to video



In [ ]:
%%bash
set -euo pipefail

python "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/cosmos3.py" \
  --model "$COSMOS3_SUPER_FP8" \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --prompt_file "$TRTLLM_ROOT/examples/visual_gen/models/cosmos3/prompts/v2v.json" \
  --video_path "$COSMOS3_FP8_OUTPUTS/super_t2v.mp4" \
  --output_path "$COSMOS3_FP8_OUTPUTS/super_v2v.mp4"



## 4. Verify the artifacts

Decode the two PNG files and print their dimensions:



In [ ]:
%%bash
set -euo pipefail

python - \
  "$COSMOS3_FP8_OUTPUTS/nano_t2i.png" \
  "$COSMOS3_FP8_OUTPUTS/super_t2i.png" <<'PY'
from PIL import Image
import sys

for path in sys.argv[1:]:
    with Image.open(path) as image:
        image.verify()
        print(path, image.format, image.size)
PY



Inspect and fully decode every generated video stream:



In [ ]:
%%bash
set -euo pipefail

for output in \
  "$COSMOS3_FP8_OUTPUTS/nano_t2v.mp4" \
  "$COSMOS3_FP8_OUTPUTS/nano_i2v.mp4" \
  "$COSMOS3_FP8_OUTPUTS/nano_v2v.mp4" \
  "$COSMOS3_FP8_OUTPUTS/super_t2v.mp4" \
  "$COSMOS3_FP8_OUTPUTS/super_i2v.mp4" \
  "$COSMOS3_FP8_OUTPUTS/super_v2v.mp4"; do
  ffprobe -v error \
    -show_entries stream=codec_type,codec_name,width,height,r_frame_rate,nb_frames \
    -of json \
    "$output"
  ffmpeg -v error -i "$output" -f null -
done



A successful encode/decode proves the interface and artifact contract; it is
not a claim that FP8 output quality matches BF16.

## 5. Scope

The Nano and Super FP8 checkpoints also contain an audio tower, so T2AV and
TI2AV requests run rather than being refused. Audio has not received the same
FP8 exercise as the four image/video modes above, and no FP8 audio-quality
claim is made here.

For the complete TensorRT-LLM Cosmos3 example contract, including serving,
guardrails, media dependencies, and other checkpoint families, see the
[TensorRT-LLM Cosmos3 example](https://github.com/NVIDIA/TensorRT-LLM/tree/main/examples/visual_gen/models/cosmos3).
